In [1]:
##MOST CURRENT

In [ ]:
# ==========================================================
# Exercise RAG Chatbot — BMI-aware Workout Planner (Fixed Full Script)
# Hybrid: Local Embeddings + FAISS retrieval + Gemini for generation
# Save embeddings to: OUT_PATH = r"C:\Users\Admin\Downloads\thesistwocodes\sentencetransformers"
# ==========================================================
# Requirements:
# pip install streamlit google-generativeai python-dotenv pandas sentence-transformers faiss-cpu torch
# ==========================================================

import os
import time
import random
import re
import shutil
from pathlib import Path

import streamlit as st
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from dotenv import load_dotenv

# ---------------------------
# Load environment variables
# ---------------------------
load_dotenv()  # supports .env file in project root

# ---------------------------
# CONFIG - user-specified
# ---------------------------
EXERCISE_CSV = r"C:\Users\Admin\Downloads\megaGymDataset.csv"
BMI_CSV = r"C:\Users\Admin\Downloads\bmiDataset.csv"

# Embedding model and where to save it (user requested path)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OUT_PATH = r"C:\Users\Admin\Downloads\thesistwocodes\sentencetransformers"
DENSE_MODEL_PATH = os.path.join(OUT_PATH, "dense")

# FAISS / embedding params
TOP_K = 5  # how many docs to retrieve

# Gemini config (generation only)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  # expects you to set this in your environment or .env
GEMINI_MODEL_NAME = "gemini-2.5-flash"

# ---------------------------
# Streamlit page setup
# ---------------------------
st.set_page_config(page_title="🏋️ BMI-aware Fitness Chatbot", layout="centered")
st.title("🏋️ BMI-aware Fitness Chatbot (Local RAG + Gemini Generator)")
st.markdown(
    "Local embeddings + FAISS retrieval for exercise & BMI data. Gemini (if available) is used for final text generation."
)

# ---------------------------
# Ensure embedding save folder exists and save/load model when needed
# This is wrapped in a cached resource below (load_embedder) so it happens only once per session.
# ---------------------------
def ensure_dense_model_saved(local_path: str, model_name: str):
    """
    Make sure a SentenceTransformer model is saved locally at local_path.
    If not present, download and save it.
    """
    os.makedirs(local_path, exist_ok=True)

    # Use config.json presence as quick existence test
    if os.path.exists(os.path.join(local_path, "config.json")):
        return True, "exists"

    # If folder contains other files but not a saved model, optionally clear it
    files = os.listdir(local_path)
    if files:
        # keep them if you wish; here we clear partial contents to avoid read errors
        try:
            shutil.rmtree(local_path)
            os.makedirs(local_path, exist_ok=True)
        except Exception:
            pass

    # Download + save
    try:
        st.info(f"Downloading and saving embedding model ({model_name}) — this may take a few minutes.")
    except Exception:
        pass

    try:
        model = SentenceTransformer(model_name)
        model.save(local_path)
        time.sleep(0.5)
        return True, "saved"
    except Exception as e:
        return False, f"error: {e}"

# ---------------------------
# CACHING: embedder + faiss indexes
# - load_embedder: ensures saved model and returns the SentenceTransformer object
# - build_indexes: encodes and builds FAISS indexes (cached)
# ---------------------------
@st.cache_resource
def load_embedder():
    ok, status = ensure_dense_model_saved(DENSE_MODEL_PATH, EMBED_MODEL)
    if not ok:
        raise RuntimeError(f"Failed to ensure saved dense model at {DENSE_MODEL_PATH}: {status}")
    # load model from local saved path
    model = SentenceTransformer(DENSE_MODEL_PATH)
    return model

@st.cache_resource
def build_indexes(gym_texts, bmi_texts):
    """
    Build and return (gym_index, bmi_index, embedder, gym_texts_list, bmi_texts_list).
    Cached so this runs only once per session unless inputs change.
    """
    st.write("🔁 [CACHE MISS] Building embeddings and FAISS indexes (runs once per session)...")
    embedder = load_embedder()

    # compute embeddings
    gym_emb = embedder.encode(gym_texts, convert_to_numpy=True, show_progress_bar=True)
    bmi_emb = embedder.encode(bmi_texts, convert_to_numpy=True, show_progress_bar=True)

    # convert to float32
    gym_emb = np.array(gym_emb, dtype="float32")
    bmi_emb = np.array(bmi_emb, dtype="float32")

    # create FAISS indexes
    gym_index = faiss.IndexFlatL2(gym_emb.shape[1])
    bmi_index = faiss.IndexFlatL2(bmi_emb.shape[1])

    gym_index.add(gym_emb)
    bmi_index.add(bmi_emb)

    st.write("✅ Embeddings encoded and FAISS indexes built and cached.")
    return gym_index, bmi_index, embedder

# ---------------------------
# Google Gemini setup (generation only)
# ---------------------------
if GOOGLE_API_KEY:
    try:
        genai.configure(api_key=GOOGLE_API_KEY)
        gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)
        st.success("Gemini configured for generation.")
    except Exception as e:
        gemini_model = None
        st.error(f"Failed to configure Gemini: {e}")
else:
    gemini_model = None
    st.warning("GOOGLE_API_KEY not set — Gemini generation disabled. Set GOOGLE_API_KEY in your environment or .env to enable it.")

# ---------------------------
# Utility: load CSVs safely (cached)
# ---------------------------
@st.cache_data
def load_csv(path):
    df = pd.read_csv(path)
    df = df.fillna("").astype(str)
    return df

# Load datasets (will raise FileNotFound if missing)
if not os.path.exists(EXERCISE_CSV):
    st.error(f"Exercise CSV not found: {EXERCISE_CSV}")
    st.stop()
if not os.path.exists(BMI_CSV):
    st.error(f"BMI CSV not found: {BMI_CSV}")
    st.stop()

gym_df = load_csv(EXERCISE_CSV)
bmi_df = load_csv(BMI_CSV)

st.write("Exercise Columns:", list(gym_df.columns))
st.write("BMI Columns:", list(bmi_df.columns))

# ---------------------------
# Prepare textual contexts
# ---------------------------
def combine_text(row):
    name = row.get("Title", "") or row.get("exercise_name", "")
    desc = row.get("Desc", "") or row.get("description", "")
    part = row.get("BodyPart", "") or row.get("main_muscle", "")
    equip = row.get("Equipment", "") or row.get("equipment", "")
    level = row.get("Level", "") or row.get("level", "")
    return f"Exercise: {name}\nMuscles: {part}\nDescription: {desc}\nEquipment: {equip}\nLevel: {level}"

gym_texts = gym_df.apply(lambda r: combine_text(r), axis=1).tolist()
bmi_texts = [f"Gender: {r['Gender']}, Age: {r['Age']}, BMI: {r['BMI']}, Case: {r['BMIcase']}" for _, r in bmi_df.iterrows()]

# Build/Load indexes (cached)
gym_index, bmi_index, embedder = build_indexes(gym_texts, bmi_texts)

# ---------------------------
# Helper functions: BMI, retrieval, sampling
# ---------------------------
def compute_bmi(weight_kg: float, height_cm: float):
    if weight_kg <= 0 or height_cm <= 0:
        return None
    h_m = height_cm / 100.0
    return round(weight_kg / (h_m * h_m), 1)

def get_bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 25:
        return "Normal"
    elif 25 <= bmi < 30:
        return "Overweight"
    else:
        return "Obese"

def retrieve_exercise_context(query, top_k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = np.array(q_emb, dtype="float32")
    _, idx = gym_index.search(q_emb, top_k)
    return [gym_texts[i] for i in idx[0]]

def find_bmi_case_by_embedding(bmi_val, gender=None, age=None):
    q_text = f"BMI: {bmi_val:.1f}"
    if gender:
        q_text += f", Gender: {gender}"
    if age:
        q_text += f", Age: {age}"
    q_emb = embedder.encode([q_text], convert_to_numpy=True)
    q_emb = np.array(q_emb, dtype="float32")
    _, idx = bmi_index.search(q_emb, 1)
    matched = bmi_df.iloc[idx[0][0]]
    return matched["BMIcase"], f"Gender: {matched['Gender']}, Age: {matched['Age']}, BMI: {matched['BMI']}, Case: {matched['BMIcase']}"

def pick_exercises_for_part(body_part: str, bmi_val: float, n: int = 5, avoid_high_impact=False):
    # Filter by BodyPart where possible (case-insensitive)
    candidates = gym_df[gym_df["BodyPart"].str.contains(body_part, case=False, na=False)] if body_part else gym_df
    if candidates.empty:
        candidates = gym_df
    # Difficulty filter based on BMI
    if bmi_val > 30:
        cand = candidates[candidates["Equipment"].str.contains("machine|cable|band|body", case=False, na=False) | candidates["Level"].str.contains("beginner|easy", case=False, na=False)]
        if not cand.empty:
            candidates = cand
    elif bmi_val < 18.5:
        cand = candidates[candidates["Desc"].str.contains("strength|compound|lift|press", case=False, na=False)]
        if not cand.empty:
            candidates = cand
    # sample without replacement
    n = min(n, len(candidates))
    sampled = candidates.sample(n=n, replace=False)
    return [f"- {row['Title']} ({row['BodyPart']}): {row['Desc']}" for _, row in sampled.iterrows()]

# ---------------------------
# Chat function (accepts chat_history list of (q,a))
# ---------------------------
def chat_with_bmi(question: str, weight: float, height: float, gender: str, age: int, chat_history=None):
    """
    Main function that composes a prompt including BMI info, retrieved contexts and optional chat_history,
    then sends to Gemini for generation (if available). If Gemini unavailable, returns a structured fallback.
    """
    # validate inputs
    if weight is None or height is None:
        return "Please provide weight and height."
    bmi_val = compute_bmi(weight, height)
    if bmi_val is None:
        return "Invalid weight/height — cannot compute BMI."

    bmi_cat = get_bmi_category(bmi_val)
    # context from bmi dataset
    try:
        bmi_case_name, bmi_case_text = find_bmi_case_by_embedding(bmi_val, gender, age)
    except Exception:
        bmi_case_name, bmi_case_text = bmi_cat, f"Calculated BMI: {bmi_val:.1f}, Category: {bmi_cat}"

    # Build history text if provided (use last N turns)
    history_text = ""
    if chat_history:
        # keep last 8 turns to limit prompt size
        recent = chat_history[-8:]
        history_text = "\n\n".join([f"User: {q}\nAssistant: {a}" for q, a in recent])
        history_text = f"Previous conversation:\n{history_text}\n\n"

    # Retrieve exercise contexts
    ex_contexts = retrieve_exercise_context(question, top_k=10)
    random.shuffle(ex_contexts)  # add some variability

    # Compose final prompt
    full_prompt = (
        "🧠 ROLE: You are an expert personal fitness coach and nutrition advisor.\n\n"
        f"USER PROFILE:\n- BMI: {bmi_val} ({bmi_cat})\n- Gender: {gender}\n- Age: {age}\n"
        f"- BMI dataset match: {bmi_case_text}\n\n"
        f"{history_text}"
        "INSTRUCTIONS:\n"
        "- Use the REFERENCE EXERCISES below only as factual source material. Do not simply repeat them verbatim.\n"
        "- If the user requests a multi-day plan, produce a structured plan with Day 1 / Day 2 etc., sets & reps, and substitution options.\n"
        "- Avoid repeating the same exercises on consecutive days unless necessary. Provide progressions or regressions when appropriate.\n"
        "- Keep explanations concise and practical.\n\n"
        "REFERENCE EXERCISES (use only as source):\n"
        + "\n\n".join(ex_contexts[:10]) +
        f"\n\nUSER QUESTION:\n{question}\n\nRESPONSE:"
    )

    # If Gemini available, use it
    if gemini_model is not None:
        try:
            gen_resp = gemini_model.generate_content(full_prompt)
            # gemini returns object with .text (or similar) — adjust if different
            return gen_resp.text.strip()
        except Exception as e:
            # fallback to structured reply if Gemini fails
            fallback = f"(Gemini error: {e})\n\nFallback plan based on retrieved exercises:\n"
    else:
        fallback = "(Gemini unavailable) Fallback plan based on retrieved exercises:\n"

    # Build a safe fallback: use retrieved exercises to assemble a simple plan
    try:
        # pick a focus word if present in question
        focus_match = re.search(r"(upper|lower|full|chest|arms|legs|back|core|abs)", question.lower())
        focus = focus_match.group(1) if focus_match else "full body"
        plan_days = 3
        m = re.search(r"(\d+)\s*(?:day|days)", question.lower())
        if m:
            plan_days = int(m.group(1))
        plan_parts = []
        for d in range(1, plan_days + 1):
            part = focus if focus != "full" else "Full Body"
            exercises = pick_exercises_for_part(part, bmi_val, n=5)
            plan_parts.append(f"Day {d} - {part}\n" + "\n".join(exercises))
        return fallback + "\n\n".join(plan_parts)
    except Exception as e:
        return f"Failed to create fallback plan: {e}"

# ---------------------------
# Session state: unified chat history
# ---------------------------
if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

# ---------------------------
# Streamlit UI - Inputs
# ---------------------------
st.subheader("🔹 Enter Your Details")
question = st.text_input("Question", placeholder="e.g. 'Give me a 4-day plan for legs'")
weight = st.number_input("Weight (kg)", min_value=1.0, max_value=300.0, value=70.0)
height = st.number_input("Height (cm)", min_value=50.0, max_value=250.0, value=170.0)
gender = st.radio("Gender", ["Male", "Female"])
age = st.number_input("Age", min_value=10, max_value=100, value=25)

# Generate Plan button
if st.button("💪 Generate Workout Plan"):
    if not question:
        st.warning("Please enter your question first.")
    else:
        with st.spinner("Generating your personalized plan..."):
            try:
                reply = chat_with_bmi(question, weight, height, gender, age, chat_history=st.session_state["chat_history"])
                # append to unified chat history
                st.session_state["chat_history"].append((question, reply))
                st.success("✅ Your Personalized Plan:")
                st.text_area("Response", reply, height=420)
            except Exception as e:
                st.error(f"⚠️ Error generating response: {e}")

st.markdown("---")

# ---------------------------
# Follow-up chat UI (uses same unified chat_history)
# ---------------------------
st.subheader("💬 Continue Chatting About Your Plan / Ask Follow-ups")
followup_query = st.text_input("Ask a follow-up question (context preserved):", key="followup_input", placeholder="e.g. 'Can I replace squats with goblet squats?'")

if st.button("🗣️ Ask Chatbot (Follow-up)"):
    if not followup_query:
        st.warning("Please enter a follow-up question.")
    else:
        with st.spinner("Thinking..."):
            try:
                # pass full history to maintain context
                ans = chat_with_bmi(followup_query, weight, height, gender, age, chat_history=st.session_state["chat_history"])
                st.session_state["chat_history"].append((followup_query, ans))
                st.success("✅ Reply:")
                st.text_area("Chatbot Answer", ans, height=280)
                # clear followup input box
                st.session_state["followup_input"] = ""
            except Exception as e:
                st.error(f"⚠️ Error generating follow-up reply: {e}")

st.markdown("---")

# ---------------------------
# Show conversation history
# ---------------------------
st.markdown("### 🧾 Conversation History (most recent last)")
if st.session_state["chat_history"]:
    for q, a in st.session_state["chat_history"]:
        st.markdown(f"**You:** {q}")
        st.markdown(f"**Chatbot:** {a}")
        st.markdown("---")
else:
    st.info("No conversation yet. Generate a plan or ask a question to start.")

# ---------------------------
# Optional: Button to clear history for testing
# ---------------------------
if st.button("Clear chat history"):
    st.session_state["chat_history"] = []
    st.success("Chat history cleared.")
